### BUILDING RECURRENCE MEMORY TRANSFORMER
- Building Baseline transformer for wikitext-103 
- Dataset consists of verified 100 good articles on wiki
- downloaded from huggingface 
- We will train the memory augmented transformer and the baseline decoder tranformer
- Also the suggested optimization steps of using mix precision and gradient accumalation from the paper are incorporated as well

In [1]:
!pip install torch transformers datasets tqdm

In [2]:
# base imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

In [3]:
scaler = GradScaler()

/tmp/ipython-input-4292142752.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


- defining accumulation steps for gradient accumulation

In [4]:
accumulation_steps = 8

**LOAD DATASET FROM HUGGINGFACE**

In [5]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

train_texts = dataset["train"]["text"]
valid_texts = dataset["validation"]["text"]
test_texts = dataset["test"]["text"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-v1/test-00000-of-00001.parq(…):   0%|          | 0.00/722k [00:00<?, ?B/s]

wikitext-103-v1/train-00000-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

wikitext-103-v1/train-00001-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

wikitext-103-v1/validation-00000-of-0000(…):   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

**TOKENIZATION**
- Tokenizing the data texts

In [6]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_texts(texts):

    tokens = []

    for t in texts:

        if len(t.strip()) == 0:
            continue

        ids = tokenizer.encode(t)

        tokens.extend(ids)

    return tokens


train_tokens = tokenize_texts(train_texts)
valid_tokens = tokenize_texts(valid_texts)
test_tokens = tokenize_texts(test_texts)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1059 > 1024). Running this sequence through the model will result in indexing errors


**Dataset for baseline model**

In [7]:
import torch

class LanguageModelDataset(Dataset):

    def __init__(self, tokens, block_size):

        self.tokens = tokens
        self.block_size = block_size

    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):

        chunk = self.tokens[idx : idx + self.block_size + 1]

        x = torch.tensor(chunk[:-1])
        y = torch.tensor(chunk[1:])

        return x, y

In [8]:
block_size = 512

train_dataset = LanguageModelDataset(train_tokens, block_size)
valid_dataset = LanguageModelDataset(valid_tokens, block_size)
test_dataset = LanguageModelDataset(test_tokens, block_size)

**DATALOADERS**

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8
)

**Dataset for Memory Transformer**

In [10]:

class SegmentDataset(Dataset):

    def __init__(self, tokens, segment_length, segments_per_sample):

        self.segment_length = segment_length
        self.segments_per_sample = segments_per_sample

        total_tokens = segment_length * segments_per_sample
        self.samples = len(tokens) // total_tokens

        self.tokens = tokens

    def __len__(self):
        return self.samples

    def __getitem__(self, idx):

        start = idx * self.segment_length * self.segments_per_sample

        segments = []

        for s in range(self.segments_per_sample):

            seg_start = start + s * self.segment_length

            seg = self.tokens[
                seg_start : seg_start + self.segment_length
            ]

            segments.append(torch.tensor(seg))

        return torch.stack(segments)

In [11]:
segment_length = 512
segments_per_sample = 8

train_mem_dataset = SegmentDataset(
    train_tokens,
    segment_length,
    segments_per_sample
)

train_mem_loader = DataLoader(
    train_mem_dataset,
    batch_size=4,
    shuffle=True
)

**Causal Self Attention**
- we will use this for both the baseline decoder transformer as well as the memory augmented transformer

In [12]:
class CausalSelfAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, max_seq_len):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len))
            .view(1, 1, max_seq_len, max_seq_len)
        )

    def forward(self, x):

        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        out = att @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

Feed Forward Network

In [13]:
class FeedForward(nn.Module):

    def __init__(self, embed_dim, expansion=4):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, expansion * embed_dim),
            nn.GELU(),
            nn.Linear(expansion * embed_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)


**Base Transformer Block**
- To be used by the memory augmented transformer as well

In [14]:
class TransformerBlock(nn.Module):

    def __init__(self, embed_dim, num_heads, max_seq_len):
        super().__init__()

        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, max_seq_len)

        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = FeedForward(embed_dim)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))

        return x




Decoder Only Transformer block

In [15]:

class DecoderOnlyTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        block_size,
        embed_dim=512,
        num_heads=8,
        num_layers=6
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embed_dim)

        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx):

        B, T = idx.shape

        positions = torch.arange(0, T, device=idx.device)

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(positions)

        x = tok_emb + pos_emb

        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits



**Memory augmented Transformer** 
- simplified style with memory tokens

In [16]:
class MemoryDecoderTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        block_size,
        embed_dim=512,
        num_heads=8,
        num_layers=6,
        memory_tokens=16
    ):
        super().__init__()

        self.memory_tokens = memory_tokens
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size + memory_tokens, embed_dim)

        self.memory_embedding = nn.Parameter(
            torch.randn(memory_tokens, embed_dim)
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, block_size + memory_tokens)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embed_dim)

        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx, memory=None):

        B, T = idx.shape

        tok_emb = self.token_embedding(idx)

        if memory is None:
            memory = self.memory_embedding.unsqueeze(0).expand(B, -1, -1)

        x = torch.cat([memory, tok_emb], dim=1)

        total_len = x.size(1)

        positions = torch.arange(0, total_len, device=idx.device)

        pos_emb = self.position_embedding(positions)

        x = x + pos_emb

        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)

        new_memory = x[:, :self.memory_tokens, :]
        token_outputs = x[:, self.memory_tokens:, :]

        logits = self.lm_head(token_outputs)

        return logits, new_memory


**Training Setups for both Transformers**

In [17]:
# Training setup for baseline decode only transformer
def train_decoder_transformer(model, loader, optimizer, device):

    model.train()
    optimizer.zero_grad()

    for step, (x, y) in enumerate(loader):

        x = x.to(device)
        y = y.to(device)

        with autocast():

            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()
            
# training setup for memory augmented transformer
def train_memory_transformer(model, loader, optimizer, device):

    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(loader):

        batch = batch.to(device)

        B, S, T = batch.shape

        memory = None

        with autocast():

            total_loss = 0

            for seg in range(S):

                segment = batch[:, seg, :]

                x = segment[:, :-1]
                y = segment[:, 1:]

                logits, memory = model(x, memory)

                memory = memory.detach()

                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    y.reshape(-1)
                )

                total_loss += loss

            total_loss = total_loss / accumulation_steps

        scaler.scale(total_loss).backward()

        if (step + 1) % accumulation_steps == 0:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()

**Validation Loop**

In [18]:
def evaluate(model, loader, device):

    model.eval()

    total_loss = 0
    count = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            total_loss += loss.item()
            count += 1

    return total_loss / count

**Model Initializations**

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"

baseline_model = DecoderOnlyTransformer(
    vocab_size=50257,
    block_size=512
).to(device)

memory_model = MemoryDecoderTransformer(
    vocab_size=50257,
    block_size=512,
    memory_tokens=16
).to(device)

optimizer_baseline = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=3e-4
)

optimizer_memory = torch.optim.AdamW(
    memory_model.parameters(),
    lr=3e-4
)

**Training both transformers**

In [20]:
epochs = 5

for epoch in range(epochs):

    print("Epoch:", epoch)

    train_decoder_transformer(
        baseline_model,
        train_loader,
        optimizer_baseline,
        device
    )

    train_memory_transformer(
        memory_model,
        train_mem_loader,
        optimizer_memory,
        device
    )

Epoch: 0


/tmp/ipython-input-2383874023.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

In [ ]:
torch.save(
    baseline_model.state_dict(),
    "decoder_transformer_wikitext103.pt"
)

torch.save(
    memory_model.state_dict(),
    "memory_transformer_wikitext103.pt"
)

GENERATING TEXT FROM SAVED MODELS

Text Generation for Baseline Transformer ↓

In [ ]:
def generate_text(model, start_tokens, max_new_tokens, device):

    model.eval()

    tokens = start_tokens.clone().to(device)

    with torch.no_grad():

        for _ in range(max_new_tokens):

            logits = model(tokens)

            next_token_logits = logits[:, -1, :]

            probs = torch.softmax(next_token_logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)

            tokens = torch.cat([tokens, next_token], dim=1)

    return tokens

In [ ]:
def decode_tokens(token_ids, tokenizer):

    return tokenizer.decode(
        token_ids,
        skip_special_tokens=True
    )

Text Generation for Memory Transformer

In [ ]:
def generate_text_memory(model, start_tokens, max_new_tokens, device):

    model.eval()

    tokens = start_tokens.clone().to(device)

    memory = None

    with torch.no_grad():

        for _ in range(max_new_tokens):

            logits, memory = model(tokens, memory)

            next_token_logits = logits[:, -1, :]

            probs = torch.softmax(next_token_logits, dim=-1)

            next_token = torch.multinomial(probs, 1)

            tokens = torch.cat([tokens, next_token], dim=1)

    return tokens

BLEU Evaluations

In [ ]:
import sarcebleu
def compute_bleu(model, loader, tokenizer, device, num_samples=100):

    model.eval()

    predictions = []
    references = []

    count = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)

            generated = generate_text(
                model,
                x[:, :20],        # prompt
                max_new_tokens=50,
                device=device
            )

            for i in range(generated.size(0)):

                pred_text = decode_tokens(
                    generated[i].cpu(),
                    tokenizer
                )

                ref_text = decode_tokens(
                    y[i].cpu(),
                    tokenizer
                )

                predictions.append(pred_text)
                references.append([ref_text])

                count += 1

                if count >= num_samples:
                    break

            if count >= num_samples:
                break

    bleu = sacrebleu.corpus_bleu(predictions, references)

    return bleu.score

In [ ]:
bleu_score_basline = compute_bleu(
    baseline_model,
    valid_loader,
    tokenizer,
    device
)

print("BLEU Score for baseline transformer:", bleu_score_basline)

In [ ]:
bleu_score = compute_bleu(
    memory_model,
    valid_loader,
    tokenizer,
    device
)

print("BLEU Score for memory transformer:", bleu_score)